# MIP Python Client — Getting Started

Welcome to the MIP Jupyter workspace. The `mip` client is pre-installed and connects to the MIP platform when you launch notebooks from the portal.

**Run all cells from top to bottom** to verify your connection and run a simple analysis.

| Where to go next | Path |
|------------------|------|
| Which API call to use | `docs/how-to-choose.md` |
| Example analysis | `examples/feres_analysis.ipynb` |
| Documentation | `docs/` (quickstart, API, troubleshooting) |
| Your notebooks | `scratch/` |

Every notebook object exposes `.help()` — for example `dm.help()` or `pipeline.help()`.

Reference: `docs/quickstart.md`

## Before you start

When you open this notebook from the **MIP portal**, your platform connection is configured automatically — run the cells below to verify it.

If `Client.from_env()` fails, see `docs/troubleshooting.md` or contact your platform administrator.

In [ ]:
import mip
from IPython.display import display
from mip.filters import F
from mip.preprocessing import MissingValuesHandler, OutlierWinsorizer

client = mip.Client.from_env()
catalog = client.catalog()
print("Public exports:", ", ".join(mip.__all__))

## Explore the API

Use discovery helpers before building an analysis. Objects expose `.help()` for method hints; see `docs/how-to-choose.md` for a goal → API decision guide.

Start with the catalog, then drill into one data model, its variables, and available algorithms.

In [ ]:
display(catalog.list())
display(catalog.summaries())
catalog.tree()

In [ ]:
dm = catalog.data_model("Dementia")
display(dm)
display(dm.help())
display(dm.summary())
display(dm.list_datasets())
display(dm.datasets.list())
display(dm.list_groups())
display(dm.tree())
display(dm.tree(group="Clinical", include_variables=True))

In [ ]:
display(dm.variables.search("age"))
display(dm.variables.numerical()[:5])
display(dm.variables.categorical()[:5])
display(dm.variables.tree(group="Clinical"))

age = dm.variables["Age"]
display(age)
display(age.summary())
display(age.details())
display(age.categories())

display(dm.datasets.search("ADNI"))
display(dm.variables.to_frame().head())

In [ ]:
display(client.algorithms().search("logistic"))
display(client.algorithms().to_frame().head())
display(mip.to_frame(client.algorithms().search("logistic")))

## Analysis and pipeline

In [ ]:
adni = dm.datasets["ADNI"]
age = dm.variables["Age"]
mmse = dm.variables["MMSE"]
diagnosis = dm.variables["Diagnosis"]

analysis_set = mip.AnalysisSet(
    data_model=dm,
    datasets=[adni],
    variables=[age, mmse, diagnosis],
)
display(analysis_set)
display(analysis_set.summary())

In [ ]:
pipeline = mip.Pipeline(
    analysis_set=analysis_set,
    filters=(F(age) >= 50) & F(mmse).is_not_null(),
    handle_missing=MissingValuesHandler(strategies={mmse: "mean"}),
    outlier_handling=OutlierWinsorizer(strategies={mmse: "iqr"}),
)
display(pipeline)
display(pipeline.recommend_algorithms())
display(pipeline.explain())

In [ ]:
histogram = pipeline.histogram(variable=mmse, bins=20)
display(histogram)
display(histogram.summary())

## Sklearn export

Logistic regression results can export to sklearn when the analysis result includes feature names and coefficients.

In [ ]:
# logreg_result = pipeline.logistic_regression(x=[age, mmse], y=diagnosis, positive_class="AD")
# sklearn_model = logreg_result.to_sklearn()

## Cohort Scout exploration

For AI-assisted exploration and novel analyses, see `scratch/README.md`. On turn 1, run `scratch-init`, then `python scratch/stroke_preflight.py` and `mip-algorithm-summary` before writing new scripts.

**Starter prompt:** "Read the agent exploration guide, run scratch-init, run preflight, log every bottleneck, then build one novel hypothesis from `examples/algorithm_examples.py`."

## Next steps

- **Which API to use:** `docs/how-to-choose.md`
- **Stroke analysis example:** `examples/feres_analysis.ipynb`
- **API reference:** `docs/api-reference.md`
- **Your own work:** copy a template into `scratch/` and adapt it